#### Transform Payments Data

1. Extract Date and Time from payment_timestamp and create new columns payment_date and payment_time
2. Map payment_status to contain descriptive values

    (1- Success, 2-Pending-, 3-Cancelled, 4-Failed)
3. Write transformed data to the Silver schema

In [0]:
df= spark.read.table('gizmobox_catalog_subbu.bronze.py_payments')
display(df)

##### 1. Extract Date and Time from payment_timestamp and create new columns payment_date and payment_time

In [0]:
from pyspark.sql import functions as F
df_extarcted_payments = (    
                            df         
                            .select(
                                'payment_id',
                                'order_id',
                                 F.date_format('payment_timestamp', 'yyyy-MM-dd').cast('date').alias('payment_date'),
                                 F.date_format('payment_timestamp', 'HH:mm:ss').alias('payment_time'),            
                                'payment_status',
                                'payment_method'
                            )
)
display(df_extarcted_payments)

##### 2. Map payment_status to contain descriptive values

    (1- Success, 2-Pending-, 3-Cancelled, 4-Failed)

In [0]:
df_mapped_payments = (
                            df_extarcted_payments
                            .select (
                                'payment_id',
                                'order_id',
                                'payment_date',
                                'payment_time',
                                F.when(df_extarcted_payments.payment_status == 1, 'Success')
                                 .when(df_extarcted_payments.payment_status == 2, 'Pending')
                                 .when(df_extarcted_payments.payment_status == 3, 'Cancelled')
                                 .when(df_extarcted_payments.payment_status == 4, 'Failed')
                                 .alias('payment_status'),
                                'payment_method
                                )
                            )
                    
display(df_mapped_payments)

In [0]:
from pyspark.sql import functions as f
df_mapped_payments1 = (
                            df_extarcted_payments
                            .select (
                                'payment_id',
                                'order_id',
                                'payment_date',
                                'payment_time',
                                F.when(f.col('payment_status') == 1, 'Success')
                                 .when(f.col('payment_status') == 2, 'Pending')
                                 .when(f.col('payment_status') == 3, 'Cancelled')
                                 .when(f.col('payment_status') == 4, 'Failed')
                                 .alias('payment_status'),
                                'payment_method'
                                )
                            )
                    
display(df_mapped_payments1)

##### 3.Write transformed data to the Silver schema

In [0]:
df_mapped_payments.writeTo('gizmobox_catalog_subbu.silver.py_payments').createOrReplace()

In [0]:
%sql
select * from gizmobox_catalog_subbu.silver.py_payments